# Linear Algebra

Most operations of that section act on each element of a tensor
separately: dividing a minibatch by 255 rescales every pixel on its own. A
model has to do more. To decide what an image shows, it combines many
numbers into a few, weighing some pixels more than others. Linear algebra
studies such weighted sums. It lets us write the computation of a model on a
whole minibatch as one expression, and it tells us, before we run anything,
what shape the result has and how much work it takes.

Our example is a simple classifier for small grayscale images of clothing,
like those of the Fashion-MNIST dataset that later chapters use. It sorts
them into ten categories such as shirts, sneakers, and bags. Each image has
$28 \times 28 = 784$ pixels. The classifier gives every category a score, a
weighted sum of the 784 pixel intensities plus an offset, and predicts the
category with the highest score. Training scores a minibatch of 256 images
at a time (that section), thousands of times over, so Python loops
over the numbers are out of the question.

In the notation of linear algebra, the scores take one line. Stack the
images as the rows of a matrix $\mathbf{X}$, the weights of each category as
the columns of a matrix $\mathbf{W}$, and the ten offsets (also called
biases) in a vector $\mathbf{b}$. Together, $\mathbf{W}$ and $\mathbf{b}$
hold the $784 \times 10 + 10 = 7{,}850$ parameters of the classifier. The
$256 \times 10 = 2{,}560$ scores of the minibatch are then

$$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b},$$

where $\mathbf{b}$ is added to every row (the figure). A
library evaluates the expression with a single call to a routine tuned for
the hardware.

![The scores of a linear classifier for a minibatch of 256 images and 10 categories. Row $i$ of $\mathbf{X}$ holds image $i$, column $j$ of $\mathbf{W}$ holds the weights of category $j$, and the two meet in the score $o_{ij}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/linear-algebra-matmul.svg)

We have not yet said what a matrix is, let alone a product of two. This
section builds the expression piece by piece and then asks two questions
that every model raises: how large is a vector, such as a list of
prediction errors, and what happens to a vector when a deep network
multiplies it by one matrix after another. that section introduced
tensors as arrays in code; here they acquire their mathematical meaning.
The book's notation is collected in that section.

In [ ]:
import torch

## Scalars, Vectors, Matrices, and Tensors

The objects of linear algebra differ in how many indices it takes to pick
out one of their numbers: none for a scalar, one for a vector, two for a
matrix, and more for a general tensor. In code, this count is the number of
axes of the tensor that stores the object.

### Scalars

A single number is a scalar: one pixel intensity, the offset of one
category, or the price of one home. We denote scalars by lowercase letters
such as $x$ and $y$, and the set of real numbers by $\mathbb{R}$; the
expression $x \in \mathbb{R}$, read "$x$ in $\mathbb{R}$", states that $x$
is a real number. In code, a scalar is a tensor with one element and no
axes, so its shape is the empty tuple:

In [ ]:
x = torch.tensor(3.0)
x, x.shape

### Vectors

A fixed-length list of scalars is a *vector*, and the scalars are its
elements (also called entries or components). Flattening one image into
its 784 pixel intensities gives a vector, and so do the 784 weights that
one category assigns to the pixels. We denote vectors by bold lowercase
letters such as $\mathbf{x}$ and $\mathbf{w}$, and write
$\mathbf{x} \in \mathbb{R}^n$ for a vector of $n$ real elements. By
default we write a vector as a column,

$$\mathbf{x} =\begin{bmatrix}x_{1} \\ \vdots \\ x_{n}\end{bmatrix},$$

where the scalar $x_i$ is its $i^\textrm{th}$ element. In running text we
write the same column vector as $\mathbf{x} = (x_1, \ldots, x_n)$, and we
list the entries of a matrix row the same way, as in "the rows $(0, 1, 2)$
and $(3, 4, 5)$". In code, a vector is a tensor with one axis:

In [ ]:
x = torch.arange(3, dtype=torch.float32)
x

Mathematics numbers the elements from $1$, whereas Python numbers them
from $0$. The third element $x_3$ is therefore `x[2]`:

In [ ]:
x[2]

The number of elements $n$ is the *dimensionality* of the vector. In code
it is the length of the tensor, and the only entry of its `shape`:

In [ ]:
len(x), x.shape

This is not the number of axes, which that section called the order
of a tensor (`ndim` in code): a vector in $\mathbb{R}^{784}$ has order 1 and
dimensionality 784. The word "dimension" is used loosely for both, and so
shall we.

### Matrices

A table of scalars with $m$ rows and $n$ columns is a *matrix*, written
$\mathbf{A} \in \mathbb{R}^{m \times n}$. A minibatch of 256 flattened
images is a matrix $\mathbf{X} \in \mathbb{R}^{256 \times 784}$ with one
example per row and one feature, the intensity of one pixel position, per
column; the book uses this layout throughout. We denote matrices by bold
capital letters and write $a_{ij}$ for the element in row $i$ and
column $j$:

$$\mathbf{A}=\begin{bmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{m1} & a_{m2} & \cdots & a_{mn} \\ \end{bmatrix}.$$

A matrix with as many rows as columns is *square*. In code, a matrix is a
tensor with two axes, and `reshape` arranges a vector's elements into one.
The $2 \times 3$ matrix $\mathbf{A}$ below serves as a small example for
the rest of the section:

In [ ]:
A = torch.arange(6, dtype=torch.float32).reshape(2, 3)
A

Exchanging the rows and columns of a matrix gives its *transpose*. If
$\mathbf{B} = \mathbf{A}^\top$, then $b_{ij} = a_{ji}$ for all $i$ and
$j$, so the transpose of an $m \times n$ matrix is an $n \times m$ matrix:

$$
\mathbf{A}^\top =
\begin{bmatrix}
    a_{11} & a_{21} & \dots  & a_{m1} \\
    a_{12} & a_{22} & \dots  & a_{m2} \\
    \vdots & \vdots & \ddots  & \vdots \\
    a_{1n} & a_{2n} & \dots  & a_{mn}
\end{bmatrix}.
$$

The classifier stores the weights of one category per column of
$\mathbf{W}$; its transpose $\mathbf{W}^\top$ holds the same weights with
one category per row.

In [ ]:
A.T

A square matrix that equals its transpose is *symmetric*. Symmetric
matrices record symmetric relations, such as the similarity between every
pair of examples. They are also the best-behaved matrices we will meet, in
a sense that the end of this section makes precise. The following matrix
$\mathbf{S}$ is symmetric:

In [ ]:
S = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
(S == S.T).all()

### Tensors

Images do not arrive as rows. A minibatch of 256 grayscale images is a tensor
of shape $(256, 1, 28, 28)$ and order 4. To feed the classifier, we flatten
each image into a row of 784 pixels. Random numbers between 0 and 1 stand in
for real images whose pixels have already been rescaled to $[0, 1]$, as in
that section. Only the shapes matter here:

In [ ]:
images = torch.rand(256, 1, 28, 28)
X = images.reshape(256, -1)  # One row of 784 pixels per image
images.shape, X.shape

## Elementwise Operations and Reductions

The classifier adds the offsets $\mathbf{b}$ to every row of
$\mathbf{X}\mathbf{W}$, and training averages a measure of error over all the
examples in a minibatch. The first is an elementwise operation; the second is
a *reduction*, which summarizes many numbers by one.

### Elementwise Operations

Elementwise operations and broadcasting work as described in
that section. The elementwise product of two matrices of the
same shape is called their
*Hadamard product* and written $\odot$:
$[\mathbf{A} \odot \mathbf{B}]_{ij} = a_{ij} b_{ij}$. It is not the matrix
product that appears in $\mathbf{X}\mathbf{W}$, which we define below. In
code, `*` computes the Hadamard product, so `A * A` squares every entry of
the $2 \times 3$ matrix $\mathbf{A}$, whereas the matrix product of
$\mathbf{A}$ with itself is not even defined:

In [ ]:
A * A

### Reductions

The sum of a vector, $\sum_{i=1}^n x_i$, and the sum of all elements of a
matrix, $\sum_{i=1}^m \sum_{j=1}^n a_{ij}$, are reductions; so are the mean,
the maximum, and the average error over a minibatch. Often we reduce along
one axis only. Summing a
matrix over its row index $i$ (axis 0) leaves one sum per column, $\sum_i
a_{ij}$, a vector of length $n$. Summing over its column index $j$ (axis 1)
leaves one sum per row, a vector of length $m$. Either way, the axis that we
reduce over disappears from the shape (the figure).

![Reductions of a $2 \times 3$ matrix. Summing over axis 0 leaves one sum per column, and summing over axis 1 leaves one sum per row; the reduced axis disappears from the shape. With `keepdims=True`, the row sums keep the shape $(2, 1)$, so broadcasting stretches them across the three columns and the division `A / sum_A` scales every row to sum to 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/linear-algebra-reduce-axes.svg)

The argument `axis` selects the axis to reduce; without it, the sum runs
over all elements:

In [ ]:
A.sum(), A.sum(axis=0), A.sum(axis=1)

The mean, the maximum, and the other reductions take the same `axis`
argument.

### Keeping the Reduced Axis

A reduction often feeds back into an elementwise operation with the tensor
it came from. Dividing each row of a nonnegative matrix by the row's sum
turns the row into proportions that add up to 1, which is the last step
when a classifier turns its scores into probabilities. The row sums of
$\mathbf{A}$ have shape $(2,)$, however, and broadcasting lines up shapes
from the right (that section). It would pair the two row
sums with the three columns of $\mathbf{A}$, and fail. The argument
`keepdims=True` keeps the reduced axis with length 1. The row sums then
form a $2 \times 1$ column, which broadcasting stretches across the columns
(the figure):

In [ ]:
sum_A = A.sum(axis=1, keepdims=True)
sum_A, sum_A.shape

PyTorch's own names for the arguments `axis` and `keepdims` are `dim` and
`keepdim`. Its reductions accept both spellings, and the code in this book
uses both.

Each row of the quotient now sums to 1:

In [ ]:
A / sum_A

On the minibatch $\mathbf{X}$, subtracting from each image its mean
intensity needs one mean per row, and `keepdims=True` gives the 256 means
the shape $(256, 1)$ that broadcasts against $(256, 784)$. Subtracting the
mean of each pixel position, one per column, needs no `keepdims`, because
the 784 column means have shape $(784,)$ and line up with the last axis:

In [ ]:
col_means = X.mean(axis=0)
row_means = X.mean(axis=1, keepdims=True)
col_means.shape, row_means.shape, (X - row_means).shape

For a square matrix, forgetting `keepdims` raises no error and subtracts
the wrong numbers, as Exercise 3
shows.

## Products

Elementwise operations combine numbers at the same position, and
reductions summarize them. A score needs both: multiply each pixel by its
weight, then add up the products. This combination is the dot product, and
all products of vectors and matrices are built from it.

### Dot Products

For one image $\mathbf{x} \in \mathbb{R}^{784}$ and the weights
$\mathbf{w} \in \mathbb{R}^{784}$ of one category, the score is
$\sum_{i=1}^{784} w_i x_i + b$. The sum is the *dot product* of the two
vectors. For $\mathbf{x}, \mathbf{y} \in \mathbb{R}^d$ it is

$$\mathbf{x}^\top \mathbf{y} = \sum_{i=1}^{d} x_i y_i.$$

The dot product is also called the *inner product* and written
$\langle \mathbf{x}, \mathbf{y} \rangle$. It is symmetric,
$\mathbf{x}^\top \mathbf{y} = \mathbf{y}^\top \mathbf{x}$, and linear in
each argument:
$(\alpha\mathbf{x} + \beta\mathbf{z})^\top \mathbf{y} = \alpha\,
\mathbf{x}^\top \mathbf{y} + \beta\, \mathbf{z}^\top \mathbf{y}$ for
scalars $\alpha$ and $\beta$. When $\mathbf{y}$ is the vector of all ones,
the dot product is the sum of the elements of $\mathbf{x}$; for the vector
$\mathbf{x} = (0, 1, 2)$ from above, it is 3:

In [ ]:
y = torch.ones(3)
x, y, torch.dot(x, y)

Equivalently, a dot product is a Hadamard product followed by a sum:

In [ ]:
torch.sum(x * y)

If the weights are nonnegative and sum to 1, the dot product
$\mathbf{w}^\top \mathbf{x}$ is a *weighted average* of the elements of
$\mathbf{x}$.

### Matrix--Vector Products

Now score one category for all 256 images of the minibatch. Each score is the
dot product of one row of $\mathbf{X}$ with the same weight vector
$\mathbf{w}$, and the 256 scores together form a *matrix--vector product*.
In general, write a matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$ in
terms of its rows $\mathbf{a}_1^\top, \ldots, \mathbf{a}_m^\top$, each a
row vector of length $n$. Its product with a vector
$\mathbf{x} \in \mathbb{R}^n$ is the vector of the $m$ dot products,

$$
\mathbf{A}\mathbf{x}
= \begin{bmatrix}
\mathbf{a}^\top_{1} \\
\mathbf{a}^\top_{2} \\
\vdots \\
\mathbf{a}^\top_m \\
\end{bmatrix}\mathbf{x}
= \begin{bmatrix}
 \mathbf{a}^\top_{1} \mathbf{x}  \\
 \mathbf{a}^\top_{2} \mathbf{x} \\
\vdots\\
 \mathbf{a}^\top_{m} \mathbf{x}\\
\end{bmatrix} \in \mathbb{R}^m.
$$

The product exists only if the number of columns of $\mathbf{A}$ equals
the length of $\mathbf{x}$, and it has one element per row of $\mathbf{A}$
(the figure). For the minibatch, the scores
$\mathbf{X}\mathbf{w} + b$ of one category form a vector in
$\mathbb{R}^{256}$, one score per image. The same product, fitted to one
real-valued label per example such as a price, is *linear regression*,
the subject of the next chapter.

![The matrix--vector product $\mathbf{A}\mathbf{x}$ takes one dot product per row of $\mathbf{A}$, so a $2\times3$ matrix maps a vector of length 3 to a vector of length 2. The first row gives $0 \cdot 0 + 1 \cdot 1 + 2 \cdot 2 = 5$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/linear-algebra-matvec.svg)

In code, take the matrix $\mathbf{A}$, with rows $(0, 1, 2)$ and
$(3, 4, 5)$, and the vector $\mathbf{x} = (0, 1, 2)$ from above.
<!-- d2l:prose id=linear-algebra-md-19 fw=mxnet, pytorch, jax -->

The operator `@` computes the matrix--vector product:

In [ ]:
A.shape, x.shape, A @ x

There is a second way to read the product, column by column, and the last
part of this section rests on it. Multiplication by
$\mathbf{A} \in \mathbb{R}^{m \times n}$ maps vectors in $\mathbb{R}^n$
to vectors in $\mathbb{R}^m$, and the map is *linear*:
$\mathbf{A}(\alpha\mathbf{x} + \beta\mathbf{y}) = \alpha\mathbf{A}\mathbf{x}
+ \beta\mathbf{A}\mathbf{y}$. What does such a map do to the *coordinate
vectors* $\mathbf{e}_j$, which have a 1 in position $j$ and 0 elsewhere?
It returns a column: the product
$\mathbf{A}\mathbf{e}_j$ is column $j$ of $\mathbf{A}$, and for our
$2 \times 3$ example $\mathbf{A}\mathbf{e}_2 = (1, 4)$. That settles every
other vector as well. Any $\mathbf{x}$ is a weighted sum of coordinate
vectors, $\mathbf{x} = \sum_j x_j \mathbf{e}_j$, so by linearity
$\mathbf{A}\mathbf{x} = \sum_j x_j \mathbf{A}\mathbf{e}_j$ is the same
weighted sum of the columns. For $\mathbf{x} = (0, 1, 2)$ it is
$1 \cdot (1, 4) + 2 \cdot (2, 5) = (5, 14)$, the output above. A matrix is
thus a list of the places to which it sends the coordinate vectors.

In the plane we can draw the whole map. A $2 \times 2$ matrix turns the
unit square, spanned by $\mathbf{e}_1$ and $\mathbf{e}_2$, into the
parallelogram spanned by its two columns (the figure). The
rotation by $90^\circ$, for instance, sends $\mathbf{e}_1$ to
$\mathbf{e}_2$ and $\mathbf{e}_2$ to $-\mathbf{e}_1$, so its columns are
$(0, 1)$ and $(-1, 0)$.

![A $2 \times 2$ matrix maps the unit square (dashed) to the parallelogram spanned by its columns $\mathbf{A}\mathbf{e}_1$ and $\mathbf{A}\mathbf{e}_2$. From left to right: a rotation by $90^\circ$, a scaling by 2 along $\mathbf{e}_1$ and by $1/2$ along $\mathbf{e}_2$, and a shear.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prelim-linear-maps.svg)

### Matrix--Matrix Products

Scoring all ten categories takes one matrix--vector product per category,
that is, per column of $\mathbf{W}$. Placing the ten results side by side
gives the *matrix--matrix product* $\mathbf{X}\mathbf{W}$, usually called
simply the matrix product. In general, the product of
$\mathbf{A} \in \mathbb{R}^{m \times k}$ and
$\mathbf{B} \in \mathbb{R}^{k \times n}$ is the matrix
$\mathbf{C} = \mathbf{A}\mathbf{B} \in \mathbb{R}^{m \times n}$ whose
entry $c_{ij}$ is the dot product of row $i$ of $\mathbf{A}$ with column
$j$ of $\mathbf{B}$:

$$c_{ij} = \sum_{l=1}^{k} a_{il}\, b_{lj}.$$

Column $j$ of $\mathbf{C}$ is the matrix--vector product of $\mathbf{A}$
with column $j$ of $\mathbf{B}$. The inner sizes must agree, and the order
of the factors matters: $\mathbf{B}\mathbf{A}$ need not even exist. For
the classifier, $(256 \times 784)(784 \times 10)$ gives $256 \times 10$,
and score $o_{ij}$ is the dot product of image $i$ with the weights of
category $j$ (the figure). Each column of $\mathbf{W}$ acts
as a template of 784 pixel weights, and the classifier scores how well
each image matches each template.

A small example first. Every column of $\mathbf{B}$ below is the all-ones
vector, so every column of $\mathbf{A}\mathbf{B}$ holds the row sums of
$\mathbf{A}$, 3 and 12:

In [ ]:
B = torch.ones(3, 4)
A @ B

Now the classifier itself, with small random weights in place of trained
ones. Its prediction for an image is the category with the largest score,
the position of the maximum along axis 1, which `argmax` computes:

In [ ]:
W = 0.01 * torch.randn(784, 10)  # Column j: category j
b = torch.zeros(10)              # Entry j: category j
O = X @ W + b                    # Row i: image i
O.shape, O.argmax(dim=1).shape

What did this cost? Each of the $mn$ entries of $\mathbf{A}\mathbf{B}$ is a
dot product of length $k$, which takes $k$ multiplications and $k - 1$
additions, so the product takes about $2mkn$ floating-point operations
(FLOPs). For the minibatch, that is
$2 \times 256 \times 784 \times 10 \approx 4 \times 10^6$ FLOPs. This is
modest, but it does not stay so. For square matrices of size $n$, the work
$2n^3$ grows faster than the $3n^2$ numbers involved, and a product of two
$4096 \times 4096$ matrices already takes about $1.4 \times 10^{11}$
FLOPs. How fast hardware gets through them is a question for
that section.

Two square matrices have names of their own. The *identity matrix*
$\mathbf{I}$, with ones on the diagonal and zeros elsewhere, has the
coordinate vectors as its columns and therefore changes nothing:
$\mathbf{I}\mathbf{x} = \mathbf{x}$ for every $\mathbf{x}$. The *inverse* $\mathbf{A}^{-1}$ of a square matrix
$\mathbf{A}$ is the matrix that undoes it,
$\mathbf{A}^{-1}\mathbf{A} = \mathbf{A}\mathbf{A}^{-1} = \mathbf{I}$. Not
every matrix has one. The scaling of the figure is undone by
scaling by $1/2$ and by 2, but a matrix that sends some nonzero vector to
zero has lost that vector for good (that section). A
layer of a neural network, finally, applies a linear map, adds an offset,
and then applies a nonlinear function to every element.

## Norms

Once a model makes predictions, training must measure how wrong they are,
and often also how large the weights have become. Both are questions about
the size of a vector: of the weights $\mathbf{w}$ or, for a model that
predicts one number per example, such as a price, of the errors
$\hat{\mathbf{y}} - \mathbf{y} \in \mathbb{R}^{256}$ of the predictions
$\hat{\mathbf{y}}$ against the labels $\mathbf{y}$ of a minibatch. A
*norm* answers such a question with one nonnegative number, and a norm of
the errors, or its square, is a common choice of *loss*, the quantity that
training minimizes. There are several norms, and the choice among them
matters: as a loss or as a penalty on the weights, each makes the model
learn something different.

### Length and Angle

The *$\ell_2$ norm* of a vector is its Euclidean length. It follows from
the theorem of Pythagoras, applied one coordinate at a time, and equals the
square root of the dot product of the vector with itself:

$$\|\mathbf{x}\|_2 = \sqrt{\mathbf{x}^\top \mathbf{x}} = \sqrt{\sum_{i=1}^n x_i^2}.$$

It is the default norm, so we often write $\|\mathbf{x}\|$ without the
subscript. Its square, applied to the errors, is the sum of squared
errors; divided by the number of examples, it becomes the mean squared
error, the standard loss for predicting real numbers. For
$\mathbf{x} = (3, -4)$ the $\ell_2$ norm is $\sqrt{9 + 16} = 5$.

Length also explains why the dot product serves to compare vectors. Take
two vectors $\mathbf{x}$ and $\mathbf{y}$ in the plane, at angle $\theta$,
and the triangle with sides $\mathbf{x}$, $\mathbf{y}$, and
$\mathbf{x} - \mathbf{y}$. The law of cosines states that
$\|\mathbf{x} - \mathbf{y}\|^2 = \|\mathbf{x}\|^2 + \|\mathbf{y}\|^2 -
2\|\mathbf{x}\|\|\mathbf{y}\|\cos\theta$. Writing the left side as a dot
product and expanding it gives
$(\mathbf{x} - \mathbf{y})^\top(\mathbf{x} - \mathbf{y}) =
\|\mathbf{x}\|^2 - 2\mathbf{x}^\top\mathbf{y} + \|\mathbf{y}\|^2$ instead.
Comparing the two expressions,

$$\mathbf{x}^\top \mathbf{y} = \|\mathbf{x}\| \|\mathbf{y}\| \cos\theta.$$

The dot product is thus the length of $\mathbf{x}$ times the signed length
$\|\mathbf{y}\|\cos\theta$ of the shadow that $\mathbf{y}$ casts on the
line through $\mathbf{x}$ (the figure). It is positive when the
angle between the vectors is less than $90^\circ$, zero when they are
perpendicular, and negative when the angle exceeds $90^\circ$.

![The dot product $\mathbf{x}^\top\mathbf{y} = \|\mathbf{x}\|\|\mathbf{y}\|\cos\theta$ is the length of $\mathbf{x}$ times the signed length $\|\mathbf{y}\|\cos\theta$ of the projection of $\mathbf{y}$ onto the line through $\mathbf{x}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/linear-algebra-dot.svg)

In 784 dimensions we cannot draw the angle between two images, so we read
the equation backward and use it to define the angle:

$$\cos\theta = \frac{\mathbf{x}^\top \mathbf{y}}{\|\mathbf{x}\| \|\mathbf{y}\|}.$$

This ratio is called the *cosine similarity*. It compares the directions of
two vectors and ignores their lengths: an image and a copy of it at twice
the brightness, $2\mathbf{x}$, have cosine similarity 1. Two vectors of
length 1 at angle $\theta$ have dot product $\cos\theta$, which ranges from
$1$ for vectors that point the same way to $-1$ for opposite ones
(the figure).

![Two unit vectors at angle $\theta$: their dot product equals $\cos\theta$, ranging from $1$ (aligned) through $0$ (perpendicular) to $-1$ (opposed).](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prelim-cosine.svg)

Are we entitled to do this? The definition the equation makes sense
only if the ratio lies between $-1$ and $1$, as a cosine must. It does, in
any number of dimensions.

**Proposition (Cauchy--Schwarz inequality).** For all
$\mathbf{x}, \mathbf{y} \in \mathbb{R}^d$,
$|\mathbf{x}^\top \mathbf{y}| \leq \|\mathbf{x}\| \|\mathbf{y}\|$.

**Proof.** If $\mathbf{x}$ or $\mathbf{y}$ is zero, both sides vanish.
Otherwise, let $\mathbf{u} = \mathbf{x}/\|\mathbf{x}\|$ and
$\mathbf{v} = \mathbf{y}/\|\mathbf{y}\|$, two vectors of length 1. A
squared length cannot be negative, so

$$0 \leq \|\mathbf{u} - \mathbf{v}\|^2 = \|\mathbf{u}\|^2 - 2\mathbf{u}^\top\mathbf{v} + \|\mathbf{v}\|^2 = 2 - 2\mathbf{u}^\top\mathbf{v},$$

which gives $\mathbf{u}^\top\mathbf{v} \leq 1$. The same argument applied to
$\|\mathbf{u} + \mathbf{v}\|^2$ gives $\mathbf{u}^\top\mathbf{v} \geq -1$.
Multiplying both bounds by $\|\mathbf{x}\|\|\mathbf{y}\|$ proves the
claim. $\blacksquare$

Equality holds only if $\mathbf{u} = \pm\mathbf{v}$, that is, if one
vector is a multiple of the other. In code, the cosine similarity takes
one line. The two vectors below have length 2 and dot product 2, so their
cosine similarity is $1/2$ and their angle is $\pi/3$, or $60^\circ$.
Doubling one of them changes nothing:

In [ ]:
def cosine(x, y):
    return torch.dot(x, y) / (torch.linalg.vector_norm(x) *
                              torch.linalg.vector_norm(y))

x, y = torch.tensor([1.0, 1, 1, 1]), torch.tensor([1.0, 1, 1, -1])
cosine(x, y), cosine(2 * x, y), torch.acos(cosine(x, y))

### Other Vector Norms

The $\ell_2$ norm squares the entries before adding them, so a few large
entries can outweigh all the small ones. For losses and penalties we
sometimes want a measure of size that weighs large and small entries more
evenly.

The *$\ell_1$ norm* sums the absolute values of the elements,

$$\|\mathbf{x}\|_1 = \sum_{i=1}^n |x_i|.$$

For $\mathbf{x} = (3, -4)$ it is $3 + 4 = 7$. If the coordinates count city
blocks, the $\ell_2$ norm is the distance in a straight line and the
$\ell_1$ norm the distance walked along the streets, hence the name
*Manhattan distance* (the figure).

![The $\ell_2$ norm of $\mathbf{x}$ is the length of the straight path to $\mathbf{x}$; the $\ell_1$ norm is the length of the path along the grid.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/linear-algebra-norms.svg)

In code, the $\ell_2$ and the $\ell_1$ norm of $(3, -4)$ are:

In [ ]:
x = torch.tensor([3.0, -4.0])
torch.linalg.vector_norm(x), torch.abs(x).sum()

In the error vector
$(1, -1, 1, -1, 10)$, the one large error contributes $100/104 \approx
96\%$ of the squared $\ell_2$ norm but only $10/14 \approx 71\%$ of the
$\ell_1$ norm. A loss built from squared errors is therefore dominated by
a few outliers, while a loss built from absolute errors limits their
influence. As penalties on the weights they differ as well: the squared
$\ell_2$ norm shrinks all weights toward zero, while the $\ell_1$ norm
tends to set many of them to zero, for reasons that
that section explains.

Both are special cases of the *$\ell_p$ norm*, defined for $p \geq 1$ by

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^n |x_i|^p \right)^{1/p}.$$

As $p$ grows, the largest element dominates the sum, and in the limit
$\|\mathbf{x}\|_\infty = \max_i |x_i|$.

In general, a norm is any function $\|\cdot\|$ from vectors to scalars
with three properties:

1. *Scaling*: $\|\alpha \mathbf{x}\| = |\alpha| \|\mathbf{x}\|$ for every
   scalar $\alpha$. Doubling a vector doubles its size.
2. *Triangle inequality*:
   $\|\mathbf{x} + \mathbf{y}\| \leq \|\mathbf{x}\| + \|\mathbf{y}\|$.
   A detour is never shorter than the direct path.
3. *Positivity*: $\|\mathbf{x}\| > 0$ for every $\mathbf{x} \neq
   \mathbf{0}$. Only the zero vector has size zero.

For the $\ell_2$ norm, the triangle inequality follows from the
Cauchy--Schwarz inequality, which bounds the middle term in

$$\|\mathbf{x} + \mathbf{y}\|^2 = \|\mathbf{x}\|^2 + 2\mathbf{x}^\top\mathbf{y} + \|\mathbf{y}\|^2 \leq \|\mathbf{x}\|^2 + 2\|\mathbf{x}\|\|\mathbf{y}\| + \|\mathbf{y}\|^2 = (\|\mathbf{x}\| + \|\mathbf{y}\|)^2.$$

The condition $p \geq 1$ is needed. For $p = 1/2$, the vectors $(1, 0)$
and $(0, 1)$ both have size 1, but their sum has size
$(1 + 1)^2 = 4 > 2$, which violates the triangle inequality.

### Matrix Norms

The weights of the classifier form a matrix, so we also need the size of a
matrix. The simplest answer treats the matrix as one long vector. The
*Frobenius norm* is the $\ell_2$ norm of all the entries,

$$\|\mathbf{A}\|_\textrm{F} = \sqrt{\sum_{i=1}^m \sum_{j=1}^n a_{ij}^2}.$$

For the $4 \times 9$ matrix of ones it is $\sqrt{36} = 6$:

In [ ]:
torch.linalg.matrix_norm(torch.ones(4, 9))

*Weight decay*, a common way to keep the weights of a model small, adds a
multiple of $\|\mathbf{W}\|_\textrm{F}^2$ to the loss. The Frobenius norm
is rather crude in one respect, though. It forgets that a matrix is also a
map, and a map raises a different question of size: by how much can it
stretch a vector? The rest of the section answers it.

## Eigenvectors and Eigenvalues

This part asks more of the reader than the rest of the section, and it can
be skipped on a first reading. The book needs it for the analysis of
weight decay (that section) and again in
that section.

A deep network applies one layer after another. Ignore everything but the
matrices, and let every layer use the same matrix $\mathbf{A}$, which must
be square so that the output of one layer fits the input of the next: after
$k$ layers, an input $\mathbf{x}$ has become $\mathbf{A}^k\mathbf{x}$. How
large is the result? For a scalar $a$ the answer is $a^k$, and it is
extreme even for values of $a$ close to 1: $1.1^{50} \approx 117$, while
$0.9^{50} \approx 0.005$. A 10% deviation from 1 per layer changes the size
of a signal by a factor of more than 100, in either direction, after 50
layers. For a matrix, the answer depends on the direction of $\mathbf{x}$,
and in a few special directions it reduces to the scalar case.

### Eigenvectors

A nonzero vector $\mathbf{v}$ is an *eigenvector* of a square matrix
$\mathbf{A}$, with *eigenvalue* $\lambda$, if

$$\mathbf{A}\mathbf{v} = \lambda \mathbf{v}.$$

The matrix does not turn an eigenvector; it only stretches it by the
factor $\lambda$, and reverses it if $\lambda < 0$. Along $\mathbf{v}$,
multiplying by $\mathbf{A}$ is the same as multiplying by the number
$\lambda$, so $\mathbf{A}^k\mathbf{v} = \lambda^k\mathbf{v}$. In
the figure, the scaling has eigenvectors $\mathbf{e}_1$ and
$\mathbf{e}_2$ with eigenvalues 2 and $1/2$, and the shear keeps the
direction of $\mathbf{e}_1$, with eigenvalue 1. The rotation turns every
direction, so it has no real eigenvector at all.

As a worked example, take the symmetric matrix from above,

$$\mathbf{S} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}.$$

Its eigenvectors can be found by inspection. Since $\mathbf{S}$ maps
$(1, 1)$ to $(3, 3)$ and $(-1, 1)$ to itself, these two directions are
eigenvectors, with eigenvalues 3 and 1.
the figure (left) shows what this means: $\mathbf{S}$ maps the
unit circle to an ellipse whose axes lie along the two eigenvectors, with
half-lengths 3 and 1. Every other direction is turned toward the
eigenvector with the larger eigenvalue; $\mathbf{e}_1 = (1, 0)$, for
instance, is mapped to $(2, 1)$.

![Left: $\mathbf{S}$ maps the unit circle (dashed) to an ellipse whose axes are its eigenvectors, stretched by $\lambda = 3$ and $\lambda = 1$, while $\mathbf{e}_1$ is turned to $\mathbf{S}\mathbf{e}_1 = (2, 1)$. Right: the directions of $\mathbf{S}^k\mathbf{e}_1$ for $k = 0, \ldots, 4$ approach the eigenvector $(1, 1)$ of the largest eigenvalue.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prelim-eigen.svg)

For symmetric matrices, libraries compute eigenvalues and eigenvectors
with `eigh`:

In [ ]:
S = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
eigenvalues, eigenvectors = torch.linalg.eigh(S)
eigenvalues, eigenvectors

The eigenvalues come in increasing order, and the eigenvectors are the
columns of the second result, scaled to length 1; their signs are
arbitrary, since $-\mathbf{v}$ is an eigenvector whenever $\mathbf{v}$ is.
The two eigenvectors of $\mathbf{S}$ are perpendicular. This holds for
every symmetric matrix: its eigenvalues are real, and it has a full set of
perpendicular eigenvectors, one per dimension (for a proof, see
that section). A matrix that is not symmetric need
not have either property, as the rotation shows.

### Powers of a Matrix

Eigenvectors answer the question posed at the start of
that section: how large $\mathbf{A}^k\mathbf{x}$
becomes after $k$ layers. We know what happens to an eigenvector, so the
idea is to take $\mathbf{x}$ apart into eigenvectors. Let us do this first
for $\mathbf{S}$ and $\mathbf{x} = \mathbf{e}_1$. Half the difference of the
two eigenvectors does it:
$\mathbf{e}_1 = \frac{1}{2}(1, 1) - \frac{1}{2}(-1, 1)$. Every
multiplication by $\mathbf{S}$ triples the first part and leaves the
second alone, so
$\mathbf{S}^k\mathbf{e}_1 = \frac{1}{2} 3^k (1, 1) - \frac{1}{2}(-1, 1) =
\left((3^k + 1)/2,\ (3^k - 1)/2\right)$. The iterates are $(2, 1)$,
$(5, 4)$, $(14, 13)$, and so on, and their direction approaches $(1, 1)$
(the figure, right).

Nothing in this computation was special to $\mathbf{S}$. A symmetric
$n \times n$ matrix $\mathbf{A}$ has $n$ perpendicular eigenvectors
$\mathbf{v}_1, \ldots, \mathbf{v}_n$, which we scale to length 1, and
since there is one per dimension, any vector is a sum
$\mathbf{x} = \sum_i c_i \mathbf{v}_i$. The coefficients are easy to find.
Taking the dot product of both sides with $\mathbf{v}_j$ removes every
term but one, because the eigenvectors are perpendicular, and leaves
$c_j = \mathbf{v}_j^\top \mathbf{x}$. Applying $\mathbf{A}$ to
$\mathbf{x}$ $k$ times multiplies each part by its own eigenvalue $k$
times:

$$\mathbf{A}^k \mathbf{x} = \sum_{i=1}^n c_i \lambda_i^k \mathbf{v}_i.$$

The part with the largest $|\lambda_i|$ grows fastest, so it soon
dominates: $\mathbf{A}^k\mathbf{x}$ turns toward the corresponding
eigenvector, and its length grows by a factor of about
$\max_i |\lambda_i|$ per step. Ten multiplications by $\mathbf{S}$ confirm
both predictions:

In [ ]:
norm = torch.linalg.vector_norm
v = torch.tensor([1.0, 0.0])
for _ in range(10):
    v_prev, v = v, S @ v
growth = norm(v) / norm(v_prev)
growth, v / norm(v)

The length grows by a factor of 3, the largest eigenvalue, per step, and
the direction agrees with the eigenvector
$(1, 1)/\sqrt{2} \approx (0.7071, 0.7071)$ to four decimals. Repeated
multiplication, with the vector rescaled to length 1 after each step so
that its entries stay in range, is called *power iteration*. It finds the
largest eigenvalue in absolute value and its eigenvector using nothing but
matrix--vector products, provided that this eigenvalue is strictly larger
in absolute value than all others and that the starting vector has a
nonzero component along its eigenvector.

The same decomposition answers the question that the Frobenius norm left
open. The largest ratio $\|\mathbf{A}\mathbf{x}\| / \|\mathbf{x}\|$ over
all $\mathbf{x} \neq \mathbf{0}$ is the *spectral norm* of $\mathbf{A}$.
For a symmetric matrix it is the largest $|\lambda_i|$, since
$\|\mathbf{A}\mathbf{x}\|^2 = \sum_i c_i^2 \lambda_i^2 \leq
\max_i \lambda_i^2 \sum_i c_i^2 = \max_i \lambda_i^2\, \|\mathbf{x}\|^2$,
with equality for the eigenvector of that eigenvalue. Matrices that are
not symmetric need more care (that section).

Now return to the deep network. The matrix

$$\mathbf{M} = \begin{bmatrix} 1 & 0.1 \\ 0.1 & 1 \end{bmatrix}$$

has the same eigenvectors as $\mathbf{S}$, with eigenvalues $1.1$ and
$0.9$, both within 10% of 1. We apply it 50 times to both eigenvectors at
once, stored as the columns of a matrix, and then measure the length of
each column:

In [ ]:
M = torch.tensor([[1.0, 0.1], [0.1, 1.0]])
# The eigenvectors (1, 1) and (-1, 1) of M, as columns of length 1
V = torch.tensor([[1.0, -1.0], [1.0, 1.0]]) / 2 ** 0.5
for _ in range(50):  # 50 "layers"
    V = M @ V
torch.linalg.vector_norm(V, dim=0)

After 50 layers, the part of a signal along the first eigenvector has grown
by a factor of about 117, and the part along the second has shrunk to
about 0.005 of its size, as $1.1^{50}$ and $0.9^{50}$ predict. Real
networks use a different matrix in each layer, with nonlinear functions in
between, but the effect persists: unless the layers stretch vectors by
factors close to 1, signals, and during training gradients, grow or vanish
exponentially with depth. that section shows how
careful initialization keeps these factors close to 1.

Everything in this section took the weights as given, and ours were random
numbers. What we cannot do yet is improve the weights. That requires
knowing how the error changes when a parameter changes, which is the
subject of the next section.

## Summary

Linear algebra writes the computation of a model on a whole minibatch as a few
operations on vectors and matrices: with one image per row of
$\mathbf{X}$, a linear classifier computes all its scores as
$\mathbf{O} = \mathbf{X}\mathbf{W} + \mathbf{b}$. Reductions summarize a
tensor along an axis, and `keepdims` keeps that axis so that the result
broadcasts back. A matrix product is a table of dot products: an
$m \times k$ matrix times a $k \times n$ matrix gives an $m \times n$
matrix at a cost of about $2mkn$ FLOPs. Norms measure the size of errors
and weights. The $\ell_2$ norm is the Euclidean length, and a dot product
divided by the two lengths is the cosine of the angle between the vectors.
Eigenvectors are the directions that a matrix only stretches, by a factor
$\lambda$ per multiplication, so repeated multiplication scales them by
powers of $\lambda$, which is why deep networks must keep the stretch of
their layers close to 1.

## Exercises

1. **Transposes and products.** Prove the following statements.
    1. $(\mathbf{A}\mathbf{B})^\top = \mathbf{B}^\top\mathbf{A}^\top$
       whenever the product is defined. Compare the entries of both sides
       using the equation.
    1. For every matrix $\mathbf{X}$, the matrix
       $\mathbf{X}\mathbf{X}^\top$ is symmetric. For the minibatch $\mathbf{X}$
       of this section, state its shape and what its entry in row $i$ and
       column $j$ measures.
1. **Identity and inverse.** The scaling of the figure has the
   matrix with rows $(2, 0)$ and $(0, 1/2)$, and the shear has the rows
   $(1, 1)$ and $(0, 1)$.
    1. Write down the inverse of each, check by hand that
       $\mathbf{A}^{-1}\mathbf{A} = \mathbf{I}$, and describe in words what
       each inverse does to the parallelogram in the figure.
    1. Show that the $2 \times 2$ matrix of all ones has no inverse: find a
       nonzero vector that it sends to $\mathbf{0}$, and explain why no
       matrix can undo that.
    1. The rotation by $90^\circ$ has the columns $(0, 1)$ and $(-1, 0)$.
       Guess its inverse from the picture, and verify your guess.
1. [code] **Shape prediction.** Without running code, predict the result of
   each operation below, then check your answers. The tensor `images` has
   shape $(256, 1, 28, 28)$, and `X` has shape $(256, 784)$, as in this
   section.
    1. `len(images)`, and the shapes of the sums of `images` along axes 2
       and 3 together (`axis=(2, 3)`), without and with `keepdims=True`.
       Which of the two results can divide `images` by broadcasting, and
       what does the division compute?
    1. `X - X.mean(axis=1)`. What goes wrong, and how does `keepdims` fix
       it?
    1. The same expression for a square minibatch of 784 images, with `X` of
       shape $(784, 784)$. Show that it runs without error, and write
       the entry in row $i$ and column $j$ of the result in terms of the
       entries of `X`. Why is this outcome worse than an error?
1. **A pairwise-obtuse triple.** The following questions ask how
   negative the pairwise dot products of several vectors can be.
    1. Determine whether three vectors $\mathbf{u}$, $\mathbf{v}$,
       $\mathbf{w}$ in the plane can satisfy
       $\mathbf{u}^\top\mathbf{v} < 0$, $\mathbf{v}^\top\mathbf{w} < 0$,
       and $\mathbf{u}^\top\mathbf{w} < 0$ simultaneously. Give a
       concrete numeric example or a short argument that none exists.
    1. Suppose $k \geq 2$ unit vectors
       $\mathbf{u}_1, \ldots, \mathbf{u}_k$ have the same dot product $c$
       for every pair. Expand
       $\|\mathbf{u}_1 + \cdots + \mathbf{u}_k\|_2^2$ to show that
       $c \geq -1/(k-1)$. What does this bound say about the cosine
       similarity of ten class templates that are all equally far apart?

    *Adapted from MIT 18.06,
    [problem set 1](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/22297c2a6dcf06d82e93ee4af115e91a_MIT18_06S10_pset1_s10_soln.pdf),
    Section 1.2, Problem 28.*
1. **Manhattan distance.** In midtown Manhattan, avenues and streets
   form a rectangular grid, and a pedestrian can walk only along them.
   Adjacent avenues lie about 250 m apart and adjacent streets about
   80 m apart.
    1. Two intersections differ by $\Delta a$ avenues and $\Delta s$
       streets. Write the length of the shortest walk between them and
       the straight-line distance between them as two norms of one
       displacement vector measured in meters.
    1. Prove that, for every $\mathbf{x} \in \mathbb{R}^n$,

       $$\|\mathbf{x}\|_2 \leq \|\mathbf{x}\|_1 \leq \sqrt{n}\, \|\mathbf{x}\|_2,$$

       and identify the vectors for which each bound holds with equality.
    1. Find the largest factor by which the shortest walk can exceed the
       straight-line distance, and the ratio $\Delta s / \Delta a$ at
       which this factor is attained.
1. [code] **Norms of higher-order tensors.** The vector norm functions of
   this section also accept tensors with three or more axes (in PyTorch,
   use `torch.linalg.vector_norm`; `matrix_norm` treats a tensor with
   three axes as a stack of matrices).
    1. Call the vector norm with its default arguments on a tensor with
       three axes. State in words what it computes, and confirm your
       statement by computing the same number directly from the tensor's
       entries.
    1. A model stores its parameters in several tensors of different
       shapes. Flatten each tensor and concatenate the results into one
       vector. Express the $\ell_2$ norm of that vector in terms of the
       norms of the individual tensors, and confirm the formula
       numerically for tensors of shapes $(3, 4)$, $(4,)$, and
       $(2, 3, 5)$.
1. **Matrix-product order and cost.** Consider three large matrices
   $\mathbf{A} \in \mathbb{R}^{1000 \times 50000}$,
   $\mathbf{B} \in \mathbb{R}^{50000 \times 20}$, and
   $\mathbf{C} \in \mathbb{R}^{20 \times 10000}$. You want the product
   $\mathbf{A} \mathbf{B} \mathbf{C}$. Count the FLOPs and the size of the
   intermediate matrix for $(\mathbf{A} \mathbf{B}) \mathbf{C}$ and for
   $\mathbf{A} (\mathbf{B} \mathbf{C})$, and state by what factors the two
   orders differ in work and in memory.
1. [code] **Pairwise distances.** ● Two sets of 50 vectors in 8 dimensions
   are stored as the rows of
   $\mathbf{X}, \mathbf{Y} \in \mathbb{R}^{50 \times 8}$. Compute the
   $50 \times 50$ squared Euclidean distances between every row of
   $\mathbf{X}$ and every row of $\mathbf{Y}$ in two ways, without loops.
    1. First, broadcast the differences of all pairs into a tensor of shape
       $(50, 50, 8)$ and sum over its last axis. Second, use
       $\|\mathbf{x}-\mathbf{y}\|^2 = \|\mathbf{x}\|^2 + \|\mathbf{y}\|^2 -
       2\mathbf{x}^\top\mathbf{y}$, which needs one matrix product and no
       intermediate tensor of that shape. Confirm that the two results
       agree to floating-point tolerance.
    1. Count the numbers that each version stores along the way when each
       set holds 10,000 vectors, and decide which version you would use.
    1. Explain why the matrix-product version can return slightly negative
       values, and how to correct them.

    *Adapted from Stanford CS231n,
    [Assignment 1](https://cs231n.github.io/assignments2026/assignment1/),
    k-nearest-neighbor distances, and fast.ai,
    [Deep Learning from the Foundations](https://github.com/fastai/course-v3/blob/master/files/dl-2019/notes/notes-2-8.md),
    Lesson 8.*

1. **Rotation matrices.** For an angle $\theta$, consider the rotation
   matrix

   $$\mathbf{R}_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}.$$

    1. Show that
       $\mathbf{R}_\alpha \mathbf{R}_\beta = \mathbf{R}_{\alpha + \beta}$,
       and interpret this identity as one rotation applied after another.
    1. Show that $\|\mathbf{R}_\theta \mathbf{x}\|_2 = \|\mathbf{x}\|_2$
       for every $\mathbf{x} \in \mathbb{R}^2$. State how the norm of
       $\mathbf{R}_\theta^k \mathbf{x}$ depends on $k$, and contrast this
       with repeated multiplication by the matrix $\mathbf{S}$ of this
       section.
    1. Show that $\mathbf{R}_\theta$ has no real eigenvalue unless
       $\theta$ is a multiple of $180^\circ$.
1. [code] **Power iteration.** ● Implement power iteration, which
   multiplies a random starting vector by a matrix and rescales the
   result to length 1 after every step, for the symmetric matrix
   $\mathbf{A}$ with rows $(1, 2, 3)$, $(2, 0, 4)$, and $(3, 4, 5)$.
    1. Report the first iteration at which the absolute value of the
       cosine similarity between successive iterates exceeds $0.999$.
       Compare the final vector with the eigenvector of the largest
       eigenvalue returned by `eigh`, and explain why the two may differ
       in sign.
    1. Measure the factor by which the sine of the angle between the
       iterate and that eigenvector shrinks per step. Express it in terms
       of the eigenvalues of $\mathbf{A}$, and use it to explain the
       iteration count from the first part.
    1. Run the same iteration on $\mathbf{A} - 9\mathbf{I}$, where
       $\mathbf{I}$ is the identity matrix. Which eigenvalue and
       eigenvector does it find now? What happens to the sign of
       successive iterates? Explain both.

    *Adapted from MIT 18.06,
    [problem set 1](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/22297c2a6dcf06d82e93ee4af115e91a_MIT18_06S10_pset1_s10_soln.pdf),
    Section 2.1, Problems 29–30.*

[Discussions](https://d2l.discourse.group/t/31)